# Data Cleaning


## Imports & Load Raw Data

In [1]:
import pandas as pd

In [2]:
df_saas = pd.read_csv("../data/raw/saas_sales.csv")
df_gdp = pd.read_csv("../data/raw/gdp_per_capita.csv")                                                                                                                                                                                                              
df_net = pd.read_csv("../data/raw/internet_penetration.csv")
                                                                                                                                                                                                                                                                      
df_saas.shape, df_gdp.shape, df_net.shape 

((9994, 19), (262, 2), (246, 2))

## 1. Check Data Types

In [ ]:
df_saas.dtypes


Row ID            int64
Order ID         object
Order Date       object
Date Key          int64
Contact Name     object
Country          object
City             object
Region           object
Subregion        object
Customer         object
Customer ID       int64
Industry         object
Segment          object
Product          object
License          object
Sales           float64
Quantity          int64
Discount        float64
Profit          float64
dtype: object

## 2. Fix Data Types

In [ ]:
# Order Date convert to datetime for filtering year/date, plotting
df_saas['Order Date'] = pd.to_datetime(df_saas['Order Date'])
df_saas.dtypes

Row ID                   int64
Order ID                object
Order Date      datetime64[ns]
Date Key                 int64
Contact Name            object
Country                 object
City                    object
Region                  object
Subregion               object
Customer                object
Customer ID              int64
Industry                object
Segment                 object
Product                 object
License                 object
Sales                  float64
Quantity                 int64
Discount               float64
Profit                 float64
dtype: object

## 3. Check for Missing Values

In [5]:
df_saas.isna().sum()

Row ID          0
Order ID        0
Order Date      0
Date Key        0
Contact Name    0
Country         0
City            0
Region          0
Subregion       0
Customer        0
Customer ID     0
Industry        0
Segment         0
Product         0
License         0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64

In [6]:
# NOTE: no missing values

## 4. Check for Duplicates

In [7]:
df_saas.duplicated().sum()

np.int64(0)

In [8]:
# NOTE: no duplicates

## 5. Standardize Country Names for Merge

In [10]:
saas_countries = df_saas['Country'].unique()
print("Saas-Countries:", len(saas_countries))
print(sorted(saas_countries))


Saas-Countries: 48
['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Czech Republic', 'Denmark', 'Egypt', 'Finland', 'France', 'Germany', 'Greece', 'Iceland', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Philippines', 'Poland', 'Portugal', 'Qatar', 'Russia', 'Saudi Arabia', 'Singapore', 'Slovenia', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Taiwan', 'Turkey', 'Ukraine', 'United Arab Emirates', 'United Kingdom', 'United States']


In [11]:
# Check which SaaS countries are NOT in the World Bank data
gdp_countries = set(df_gdp['Country'])
net_countries = set(df_net['Country'])

no_match_gdp = [c for c in saas_countries if c not in gdp_countries]
no_match_net = [c for c in saas_countries if c not in net_countries]

print("No GDP match:", no_match_gdp)
print("No Internet match:", no_match_net)

No GDP match: ['Turkey', 'Russia', 'Czech Republic', 'South Korea', 'Egypt', 'Taiwan']
No Internet match: ['Turkey', 'Russia', 'Czech Republic', 'South Korea', 'Egypt', 'Taiwan']


In [12]:
# Find the World Bank names for mismatched countries
search_terms = ["Turk", "Russia", "Czech", "Korea", "Egypt", "Taiwan"]

for term in search_terms:
    matches = df_gdp[df_gdp['Country'].str.contains(term, case=False)]['Country'].values
    print(f"{term}: {matches}")

Turk: ['Turkiye' 'Turkmenistan' 'Turks and Caicos Islands']
Russia: ['Russian Federation']
Czech: ['Czechia']
Korea: ['Korea, Rep.']
Egypt: ['Egypt, Arab Rep.']
Taiwan: []


## 6. Rename Countries for Merge

In [13]:
# Mapping SaaS names to World Bank names
country_mapping = {
    "Turkey":"Turkiye",
    "Russia":"Russian Federation",
    "Czech Republic":"Czechia",
    "South Korea":"Korea, Rep.",
    "Egypt":"Egypt, Arab Rep."
}

df_saas['Country'] = df_saas['Country'].replace(country_mapping)

In [ ]:
# NOTE: Sorry Taiwan!

## 7. Merge Datasets

In [14]:
# Merge SaaS data with GDP and Internet Penetration
df = df_saas.merge(df_gdp, on='Country', how="left")
df = df.merge(df_net, on='Country', how="left")

# Check how many rows lost data
print("Missing GDP:", df['GDP_per_Capita'].isnull().sum())
print("Missing Internet:", df['Internet_Penetration'].isnull().sum())
df.shape

Missing GDP: 21
Missing Internet: 21


(9994, 21)

In [15]:
# NOTE: why left -> all SaaS-Transactions remain, even if no match in World Bank (eg. Taiwan); inner merge all rows deleted
# Missing values -> Taiwan transactions

## 8. Create Calculated Column

In [16]:
# Calculate profit margin as percentage
df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100
df[['Sales', 'Profit', 'Profit Margin']].head()

,Sales,Profit,Profit Margin
0,261.9600,41.9136,16.00
1,731.9400,219.5820,30.00
2,14.6200,6.8714,47.00
3,957.5775,-383.0310,-40.00
4,22.3680,2.5164,11.25


In [ ]:
# NOTE: Round to 2 dec and Profit_Margin as int? -> ask Cristina what is industry standard?

## Save Clean Data

In [21]:
# Round financial columns to 2 decimal places
df["Sales"] = df["Sales"].round(2)                        
df["Profit"] = df["Profit"].round(2)                                                                                                                                                                                                                                
df["Profit Margin"] = df["Profit Margin"].round(2)

# Save cleaned data
df.to_csv("../data/cleaned/saas_sales_clean.csv", index=False)
df.shape

(9994, 22)

In [22]:
df.head()

,Row ID,Order ID,Order Date,Date Key,Contact Name,Country,City,Region,Subregion,Customer,...,Segment,Product,License,Sales,Quantity,Discount,Profit,GDP_per_Capita,Internet_Penetration,Profit Margin
0,1,EMEA-2022-152156,2022-11-09,20221109,Nathan Bell,Ireland,Dublin,EMEA,UKIR,Chevron,...,SMB,Marketing Suite,16GRM07R1K,261.96,2,0.00,41.91,112894.953241,96.4974,16.00
1,2,EMEA-2022-152156,2022-11-09,20221109,Nathan Bell,Ireland,Dublin,EMEA,UKIR,Chevron,...,SMB,FinanceHub,QLIW57KZUV,731.94,3,0.00,219.58,112894.953241,96.4974,30.00
2,3,AMER-2022-138688,2022-06-13,20220613,Deirdre Bailey,United States,New York City,AMER,NAMER,Phillips 66,...,Strategic,FinanceHub,JI6BVL70HQ,14.62,2,0.00,6.87,84534.040784,93.1444,47.00
3,4,EMEA-2021-108966,2021-10-11,20211011,Zoe Hodges,Germany,Stuttgart,EMEA,EU-WEST,Royal Dutch Shell,...,SMB,ContactMatcher,DE9GJKGD44,957.58,5,0.45,-383.03,56103.732318,93.5000,-40.00
4,5,EMEA-2021-108966,2021-10-11,20211011,Zoe Hodges,Germany,Stuttgart,EMEA,EU-WEST,Royal Dutch Shell,...,SMB,Marketing Suite - Gold,OIF7NY23WD,22.37,2,0.20,2.52,56103.732318,93.5000,11.25


## Summary                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                      
  ### Dataset Overview                                                                                                                                                                                                                                                
  | | Raw | Clean |                                                                                                                                                                                                                                                   
  |---|---|---|                                                                                                                                                                                                                                                       
  | Rows | 9,994 | 9,994 |                                                                                                                                                                                                                                            
  | Columns | 20 | 21 |   
                                                                                                                                                                                                                                                                      
  ### Cleaning Steps Applied                                
  | # | Technique | Result |                                                                                                                                                                                                                                          
  |---|---|---|             
  | 1 | Check data types | 1 issue found (Order Date) |                                                                                                                                                                                                               
  | 2 | Fix data types | Order Date: object → datetime |    
  | 3 | Check missing values | None found |                                                                                                                                                                                                                           
  | 4 | Check duplicates | None found |                                                                                                                                                                                                                               
  | 5 | Standardize country names | 5 countries renamed |                                                                                                                                                                                                             
  | 6 | Merge with World Bank data | GDP + Internet Penetration added |                                                                                                                                                                                               
  | 7 | Create calculated column | Profit Margin added |                                                                                                                                                                                                              
                                                                                                                                                                                                                                                                      
  ### Notes                                                                                                                                                                                                                                                           
  | Issue | Detail |                                                                                                                                                                                                                                                  
  |---|---|                                                 
  | Taiwan | No World Bank data available (21 transactions affected) |
  | Rounding | Sales, Profit, Profit Margin rounded to 2 decimals | 